In [39]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import cv2
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
layers = tf.keras.layers

Load the data and process

In [4]:
df = pd.read_csv("./dataset/labels.csv")

In [7]:
def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        print("Bad image:", path)
        return None

    img = cv2.resize(img, (224, 224))
    img = img.astype("float32") / 255.0
    #img = np.stack([img, img, img], axis=-1)
    return img

In [8]:
images = []
labels = []

for path, label in tqdm(zip(df["image"], df["label"]), total=len(df)):
    img = load_image(path)
    if img is None:
        continue
    images.append(img)
    labels.append(label)

X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.float32)

100%|██████████| 1734/1734 [04:13<00:00,  6.83it/s]


In [11]:
X = np.expand_dims(X, axis=-1)

Split the data

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

Check the data

In [13]:
# Verify the correct format
print(X_train.shape)
print(X_test.shape)

(1387, 224, 224, 1)
(347, 224, 224, 1)


In [29]:
# Flatten data for other techniques
X_train_flat = X_train.reshape(1387, -1)
X_test_flat = X_test.reshape(347, -1)
print(X_train_flat.shape)
print(X_test_flat.shape)

(1387, 50176)
(347, 50176)


## First Technique: CNN

In [14]:
# Model
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(224, 224, 1)),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(1, activation='sigmoid')  # binary classification
])

In [15]:
# Verify the model is correct
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     5,537,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,630,593 (21.48 MB)

 Trainable params: 5,630,593 (21.48 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [17]:
# Fit full dataset
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=8,
    batch_size=32
)

Epoch 1/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 34s 754ms/step - accuracy: 0.5494 - loss: 0.6575 - val_accuracy: 0.5648 - val_loss: 0.6440
Epoch 2/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 33s 745ms/step - accuracy: 0.5984 - loss: 0.6206 - val_accuracy: 0.6023 - val_loss: 0.6154
Epoch 3/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 32s 736ms/step - accuracy: 0.6943 - loss: 0.5799 - val_accuracy: 0.6686 - val_loss: 0.5977
Epoch 4/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 33s 757ms/step - accuracy: 0.7289 - loss: 0.5497 - val_accuracy: 0.6945 - val_loss: 0.5791
Epoch 5/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 33s 739ms/step - accuracy: 0.7340 - loss: 0.5449 - val_accuracy: 0.7118 - val_loss: 0.5930
Epoch 6/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 39s 880ms/step - accuracy: 0.7368 - loss: 0.5295 - val_accuracy: 0.6888 - val_loss: 0.5919
Epoch 7/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 37s 835ms/step - accuracy: 0.7563 - loss: 0.5097 - val_accuracy: 0.7061 - val_loss: 0.5701
Epoch 8/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 34s 772ms/step - accuracy: 0.7570 - loss: 0.4981 - val_accuracy: 0.

## Second Technique: KNN

In [26]:
knn = KNeighborsClassifier(n_neighbors=10, n_jobs=-1)
knn.fit(X_train_flat, y_train)

,n_neighbors,10
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,-1


In [ ]:
y_pred_knn = knn.predict(X_test_flat)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_knn))

Accuracy: 0.7463976945244957


## Third Technique: SVM

In [34]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

In [35]:
svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [36]:
y_pred_svm = svm.predict(X_test_scaled)

In [37]:
print("Accuracy:", accuracy_score(y_test, y_pred_svm))

Accuracy: 0.7002881844380403


## Fourth Technique: Random Forests

In [40]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train_flat, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [41]:
y_pred_rf = rf.predict(X_test_flat)

In [42]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

Accuracy: 0.7982708933717579
